In [1]:
# ==============================================================================
# CELL 1: Mount Google Drive and Import Libraries
# ==============================================================================

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from pathlib import Path
import warnings
from tqdm import tqdm

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✓ Cell 1 Complete: Libraries imported")

Mounted at /content/drive
✓ Cell 1 Complete: Libraries imported


In [2]:
# ==============================================================================
# CELL 2: Define Paths and Setup
# ==============================================================================

DATA_PATH = Path("/content/drive/MyDrive/medbot")
HOSP_PATH = DATA_PATH / "hosp"
ICU_PATH = DATA_PATH / "icu"
OUTPUT_PATH = Path("output")
OUTPUT_PATH.mkdir(exist_ok=True)

print("=" * 80)
print("MIMIC-IV Demo Dataset Preprocessing Pipeline")
print("=" * 80)
print(f"\nData Path: {DATA_PATH}")
print(f"Output Path: {OUTPUT_PATH}")
print("\nDataset Overview:")
print("- 100 demo patients from MIMIC-IV")
print("- Hospital (hosp) module: Patient demographics, admissions, diagnoses, procedures, prescriptions, labs, etc.")
print("- ICU module: ICU stays, chartevents, inputevents, outputevents, procedures, etc.")
print("\n✓ Cell 2 Complete: Paths configured")


MIMIC-IV Demo Dataset Preprocessing Pipeline

Data Path: /content/drive/MyDrive/medbot
Output Path: output

Dataset Overview:
- 100 demo patients from MIMIC-IV
- Hospital (hosp) module: Patient demographics, admissions, diagnoses, procedures, prescriptions, labs, etc.
- ICU module: ICU stays, chartevents, inputevents, outputevents, procedures, etc.

✓ Cell 2 Complete: Paths configured


In [3]:
# ==============================================================================
# CELL 3: Load Core Patient and Admission Data
# ==============================================================================

print("\n" + "="*80)
print("STEP 1: Loading Core Patient and Admission Data")
print("="*80)

# Load patient demographics
print(f"Loading patients...", end=" ")
patients = pd.read_csv(HOSP_PATH / "patients.csv.gz")
print(f"\n✓ Patients loaded: {len(patients):,} records, {patients['subject_id'].nunique()} unique patients")

# Load admissions
print(f"Loading admissions...", end=" ")
admissions = pd.read_csv(HOSP_PATH / "admissions.csv.gz")
print(f"✓ Admissions loaded: {len(admissions):,} records, {admissions['hadm_id'].nunique()} unique admissions")

# Convert datetime columns
datetime_cols_admissions = ['admittime', 'dischtime', 'deathtime', 'edregtime', 'edouttime']
for col in datetime_cols_admissions:
    admissions[col] = pd.to_datetime(admissions[col])

# Calculate length of stay
admissions['los_days'] = (admissions['dischtime'] - admissions['admittime']).dt.total_seconds() / (24 * 3600)

print(f"\nAdmission Statistics:")
print(f"- Average LOS: {admissions['los_days'].mean():.2f} days")
print(f"- Median LOS: {admissions['los_days'].median():.2f} days")
print(f"- Hospital mortality rate: {admissions['hospital_expire_flag'].mean()*100:.2f}%")
print("\n✓ Cell 3 Complete: Patient and admission data loaded")


STEP 1: Loading Core Patient and Admission Data
Loading patients... 
✓ Patients loaded: 364,627 records, 364627 unique patients
Loading admissions... ✓ Admissions loaded: 546,028 records, 546028 unique admissions

Admission Statistics:
- Average LOS: 4.76 days
- Median LOS: 2.82 days
- Hospital mortality rate: 2.16%

✓ Cell 3 Complete: Patient and admission data loaded


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
 # ==============================================================================
# CELL 4: Load and Process Diagnoses Data (Stream-to-Disk)
# ==============================================================================
import gc
print("\n" + "="*80)
print("STEP 2: Processing Diagnoses Data (Stream-to-Disk)")
print("="*80)

print(f"Loading d_icd_diagnoses dictionary...", end=" ")
d_icd_diagnoses = pd.read_csv(HOSP_PATH / "d_icd_diagnoses.csv.gz")
print(f"✓ Loaded {len(d_icd_diagnoses):,} codes")

# Prepare output file
diagnoses_output_file = OUTPUT_PATH / "mimic_diagnoses_detailed.csv"
print(f"Streaming detailed diagnoses to: {diagnoses_output_file}")

# Comorbidity definitions for on-the-fly extraction
comorbidity_cols = [
    "hypertension", "diabetes_type1", "diabetes_type2", "cad_hf",
    "aki", "ckd", "chronic_liver_disease", "copd_asthma",
    "malignancy", "immunosuppressed"
]
comorbidity_chunks = []
total_diagnoses = 0

chunk_size = 100000
first_chunk = True

print(f"Processing diagnoses_icd in chunks of {chunk_size}...")

with pd.read_csv(HOSP_PATH / "diagnoses_icd.csv.gz", chunksize=chunk_size, iterator=True) as reader:
    for chunk in reader:
        total_diagnoses += len(chunk)

        # 1. Merge for Detailed Output
        chunk_merged = chunk.merge(d_icd_diagnoses, on=['icd_code', 'icd_version'], how='left')

        # Write to disk
        chunk_merged.to_csv(diagnoses_output_file,
                          mode='w' if first_chunk else 'a',
                          header=first_chunk,
                          index=False)
        first_chunk = False

        # 2. Extract Comorbidities (On-the-fly)
        # Create lightweight view for this chunk
        c_df = chunk[['hadm_id', 'icd_code']].copy()
        c_df['icd_code'] = c_df['icd_code'].astype(str)

        # Vectorized Flagging
        c_df["hypertension"] = c_df["icd_code"].str.startswith(("401", "I10"))
        c_df["diabetes_type1"] = c_df["icd_code"].str.startswith(("250.1", "E10"))
        c_df["diabetes_type2"] = c_df["icd_code"].str.startswith(("250.0", "E11"))
        c_df["cad_hf"] = c_df["icd_code"].str.startswith(("414", "428", "I25", "I50"))
        c_df["aki"] = c_df["icd_code"].str.startswith(("584", "N17"))
        c_df["ckd"] = c_df["icd_code"].str.startswith(("585", "N18"))
        c_df["chronic_liver_disease"] = c_df["icd_code"].str.startswith(("571", "K70", "K74"))
        c_df["copd_asthma"] = c_df["icd_code"].str.startswith(("493", "J44", "J45"))
        c_df["malignancy"] = c_df["icd_code"].str.startswith(("C", "140"))
        c_df["immunosuppressed"] = c_df["icd_code"].str.startswith(("V58.6", "Z79.8"))

        # Aggregate chunk to 1 row per admission (partial result)
        chunk_summary = c_df.groupby("hadm_id")[comorbidity_cols].any().reset_index()
        comorbidity_chunks.append(chunk_summary)

        # cleanup
        del chunk, chunk_merged, c_df, chunk_summary

print(f"✓ Streamed {total_diagnoses:,} diagnosis records")

# Aggregating final comorbidity flags
print("Aggregating final comorbidity flags from chunks...")
if comorbidity_chunks:
    comorbidity_concat = pd.concat(comorbidity_chunks, ignore_index=True)
    # Group by hadm_id one last time to combine partial results from different chunks
    comorbidity_summary = comorbidity_concat.groupby("hadm_id")[comorbidity_cols].any().reset_index()
    del comorbidity_concat, comorbidity_chunks
else:
    comorbidity_summary = pd.DataFrame(columns=['hadm_id'] + comorbidity_cols)

gc.collect()

# We count 'total diagnoses per hadm' roughly for stats, but to save RAM we won't keep the full counts in memory
# unless needed. The original script printed stats. Let's skip detailed stats for now or assume user cares about output.
print(f"✓ Comorbidity flags calculated for {len(comorbidity_summary):,} admissions")
print("\n✓ Cell 4 Complete: Diagnoses processed and streamed")


STEP 2: Processing Diagnoses Data (Stream-to-Disk)
Loading d_icd_diagnoses dictionary... ✓ Loaded 112,107 codes
Streaming detailed diagnoses to: output/mimic_diagnoses_detailed.csv
Processing diagnoses_icd in chunks of 100000...
✓ Streamed 6,364,488 diagnosis records
Aggregating final comorbidity flags from chunks...
✓ Comorbidity flags calculated for 545,497 admissions

✓ Cell 4 Complete: Diagnoses processed and streamed


In [6]:
# ==============================================================================
# CELL 5: Load and Process Procedures Data (Stream-to-Disk)
# ==============================================================================

print("\n" + "="*80)
print("STEP 3: Processing Procedures Data (Stream-to-Disk)")
print("="*80)

print(f"Loading d_icd_procedures dictionary...", end=" ")
d_icd_procedures = pd.read_csv(HOSP_PATH / "d_icd_procedures.csv.gz")
print(f"✓ Loaded {len(d_icd_procedures):,} codes")

# Output file
procedures_output_file = OUTPUT_PATH / "mimic_procedures_detailed.csv"
print(f"Streaming detailed procedures to: {procedures_output_file}")

# Dialysis logic: ICD codes starting with 39.95 or 54.98 (ICD-9 based logic from original script)
# We need to collect hadm_ids that have these codes
dialysis_hadm_ids = set()
total_procedures = 0
first_chunk = True

print(f"Processing procedures_icd in chunks of {chunk_size}...")

with pd.read_csv(HOSP_PATH / "procedures_icd.csv.gz", chunksize=chunk_size, iterator=True) as reader:
    for chunk in reader:
        total_procedures += len(chunk)

        # 1. Merge for Output
        chunk_merged = chunk.merge(d_icd_procedures, on=['icd_code', 'icd_version'], how='left')
        chunk_merged['chartdate'] = pd.to_datetime(chunk_merged['chartdate'])

        chunk_merged.to_csv(procedures_output_file,
                          mode='w' if first_chunk else 'a',
                          header=first_chunk,
                          index=False)
        first_chunk = False

        # 2. Extract Dialysis Flags
        # ICD-9 codes 39.95 (Hemodialysis) and 54.98 (Peritoneal dialysis)
        # Check carefully if ICD-10 is used in dataset, typically MIMIC-IV has mixed versions.
        # Original script used: .str.startswith(("39.95", "54.98"))

        chunk_dialysis = chunk[chunk['icd_code'].astype(str).str.startswith(("39.95", "54.98"))]
        if not chunk_dialysis.empty:
            dialysis_hadm_ids.update(chunk_dialysis['hadm_id'].tolist())

        del chunk, chunk_merged, chunk_dialysis

print(f"✓ Streamed {total_procedures:,} procedure records")

# Create simple dataframe for dialysis flags
dialysis_flags = pd.DataFrame({'hadm_id': list(dialysis_hadm_ids)})
dialysis_flags['on_dialysis'] = True

print(f"✓ Dialysis flags calculated for {len(dialysis_flags):,} admissions")
print("\n✓ Cell 5 Complete: Procedures processed and streamed")



STEP 3: Processing Procedures Data (Stream-to-Disk)
Loading d_icd_procedures dictionary... ✓ Loaded 86,423 codes
Streaming detailed procedures to: output/mimic_procedures_detailed.csv
Processing procedures_icd in chunks of 100000...
✓ Streamed 859,655 procedure records
✓ Dialysis flags calculated for 0 admissions

✓ Cell 5 Complete: Procedures processed and streamed


In [7]:
# ==============================================================================
# CELL 6: Load and Process Prescription Data (Stream-to-Disk)
# ==============================================================================

print("\n" + "="*80)
print("STEP 4: Processing Prescription Data (Stream-to-Disk)")
print("="*80)

# Output file
prescriptions_output_file = OUTPUT_PATH / "mimic_prescriptions.csv"
print(f"Streaming prescriptions to: {prescriptions_output_file}")

# Vasopressor logic
vasopressor_drugs = ["norepinephrine", "epinephrine", "dopamine", "vasopressin", "phenylephrine"]
pressor_hadm_ids = set()
total_prescriptions = 0
first_chunk = True

print(f"Processing prescriptions in chunks of {chunk_size}...")

with pd.read_csv(HOSP_PATH / "prescriptions.csv.gz", chunksize=chunk_size, iterator=True) as reader:
    for chunk in reader:
        total_prescriptions += len(chunk)

        # 1. Process DateTimes for Output
        chunk['starttime'] = pd.to_datetime(chunk['starttime'])
        chunk['stoptime'] = pd.to_datetime(chunk['stoptime'])
        chunk['duration_hours'] = (chunk['stoptime'] - chunk['starttime']).dt.total_seconds() / 3600

        chunk.to_csv(prescriptions_output_file,
                   mode='w' if first_chunk else 'a',
                   header=first_chunk,
                   index=False)
        first_chunk = False

        # 2. Extract Vasopressor Flags
        # Check drug column (case-insensitive)
        chunk_pressor = chunk[chunk['drug'].astype(str).str.lower().isin(vasopressor_drugs)]
        if not chunk_pressor.empty:
            pressor_hadm_ids.update(chunk_pressor['hadm_id'].tolist())

        del chunk, chunk_pressor

print(f"✓ Streamed {total_prescriptions:,} prescription records")

# Create simple dataframe for pressor flags
pressor_flags = pd.DataFrame({'hadm_id': list(pressor_hadm_ids)})
pressor_flags['on_vasopressors'] = True

print(f"✓ Vasopressor flags calculated for {len(pressor_flags):,} admissions")
print("\n✓ Cell 6 Complete: Prescriptions processed and streamed")


STEP 4: Processing Prescription Data (Stream-to-Disk)
Streaming prescriptions to: output/mimic_prescriptions.csv
Processing prescriptions in chunks of 100000...
✓ Streamed 20,292,611 prescription records
✓ Vasopressor flags calculated for 35,152 admissions

✓ Cell 6 Complete: Prescriptions processed and streamed


In [8]:
# ==============================================================================
# CELL 7: Load and Process Lab Events Data (Optimized)
# ==============================================================================

print("\n" + "="*80)
print("STEP 5: Loading and Processing Lab Events Data (Chunked & Column-Optimized)")
print("="*80)

# Define key lab test itemids (Moved from later cells for early filtering)
key_lab_items = {
    50868: 'Anion Gap',
    50882: 'Bicarbonate',
    50893: 'Calcium Total',
    50902: 'Chloride',
    50912: 'Creatinine',
    50931: 'Glucose',
    50960: 'Magnesium',
    50970: 'Phosphate',
    50971: 'Potassium',
    50983: 'Sodium',
    51006: 'Urea Nitrogen',
    51221: 'Hematocrit',
    51222: 'Hemoglobin',
    51265: 'Platelet Count',
    51279: 'Red Blood Cells',
    51301: 'White Blood Cells'
}

liver_lab_map = {
    "alt": [50861],
    "ast": [50878],
    "alp": [50863],
    "total_bilirubin": [50885]
}

# Collect all itemids we care about to filter during load
relevant_lab_items = set(key_lab_items.keys())
for item_list in liver_lab_map.values():
    relevant_lab_items.update(item_list)

print(f"Defining {len(relevant_lab_items)} relevant lab items for filtering.")

# Load lab events with chunking
print(f"Loading labevents (chunked)...", end=" ")
chunk_size = 100000  # Reduced to 100k
lab_chunks = []
total_rows_scanned = 0

# Cols to keep
use_cols = ['hadm_id', 'itemid', 'charttime', 'valuenum', 'valueuom']
dtypes = {
    'itemid': 'int32',
    'hadm_id': 'float32',  # hadm_id can be NaN, so must be float or Int32
    'valuenum': 'float32'
}

# Use chunking to avoid loading massive file into memory
with pd.read_csv(HOSP_PATH / "labevents.csv.gz",
                 chunksize=chunk_size,
                 usecols=use_cols,
                 # Note: hadm_id as float because it has NaNs in raw file sometimes
                 iterator=True) as reader:
    for chunk in reader:
        total_rows_scanned += len(chunk)
        # Filter for only the items we need
        filtered_chunk = chunk[chunk['itemid'].isin(relevant_lab_items)]
        if not filtered_chunk.empty:
            lab_chunks.append(filtered_chunk)
        # Explicit garbage collection hint
        del chunk

# Concatenate only the useful data
if lab_chunks:
    labevents = pd.concat(lab_chunks, ignore_index=True)
else:
    labevents = pd.DataFrame(columns=use_cols)

print(f"\n✓ Lab events loaded & filtered: {len(labevents):,} relevant records (from ~{total_rows_scanned:,} total)")

# Clean up memory
del lab_chunks
import gc
gc.collect()

print(f"Loading d_labitems...", end=" ")
d_labitems = pd.read_csv(HOSP_PATH / "d_labitems.csv.gz")

print(f"\n✓ Lab item dictionary loaded: {len(d_labitems):,} unique lab tests")

# Merge with lab item descriptions
labevents_merged = labevents.merge(
    d_labitems,
    on='itemid',
    how='left'
)

# Convert datetime
labevents_merged['charttime'] = pd.to_datetime(labevents_merged['charttime'])

# Count lab tests per admission (Note: This is now only KEY lab tests)
labs_per_admission = labevents_merged.groupby('hadm_id').size().reset_index(name='num_lab_tests')

print(f"\nLab Events Statistics (Filtered):")
print(f"- Total unique lab test types: {labevents_merged['itemid'].nunique()}")
print(f"- Total unique admissions with labs: {labevents_merged['hadm_id'].nunique()}")
print(f"- Avg lab tests per admission: {labs_per_admission['num_lab_tests'].mean():.2f}")
print("\n✓ Cell 7 Complete: Lab events data processed")


STEP 5: Loading and Processing Lab Events Data (Chunked & Column-Optimized)
Defining 20 relevant lab items for filtering.
Loading labevents (chunked)... 
✓ Lab events loaded & filtered: 68,901,139 relevant records (from ~158,374,764 total)
Loading d_labitems... 
✓ Lab item dictionary loaded: 1,650 unique lab tests

Lab Events Statistics (Filtered):
- Total unique lab test types: 20
- Total unique admissions with labs: 439023
- Avg lab tests per admission: 97.70

✓ Cell 7 Complete: Lab events data processed


In [9]:
# ==============================================================================
# CELL 8: Load ICU Stays Data
# ==============================================================================

print("\n" + "="*80)
print("STEP 6: Loading ICU Stays Data")
print("="*80)

# Load ICU stays
print(f"Loading icustays...", end=" ")
icustays = pd.read_csv(ICU_PATH / "icustays.csv.gz")

print(f"\n✓ ICU stays loaded: {len(icustays):,} records")

# Convert datetime columns
icustays['intime'] = pd.to_datetime(icustays['intime'])
icustays['outtime'] = pd.to_datetime(icustays['outtime'])

print(f"\nICU Stay Statistics:")
print(f"- Total unique ICU stays: {icustays['stay_id'].nunique()}")
print(f"- Total unique patients with ICU stays: {icustays['subject_id'].nunique()}")
print(f"- Average ICU LOS: {icustays['los'].mean():.2f} days")
print(f"- Median ICU LOS: {icustays['los'].median():.2f} days")
print("\n✓ Cell 8 Complete: ICU stays data loaded")


STEP 6: Loading ICU Stays Data
Loading icustays... 
✓ ICU stays loaded: 94,458 records

ICU Stay Statistics:
- Total unique ICU stays: 94458
- Total unique patients with ICU stays: 65366
- Average ICU LOS: 3.63 days
- Median ICU LOS: 1.97 days

✓ Cell 8 Complete: ICU stays data loaded


In [10]:
# ==============================================================================
# CELL 9: Load ICU Chart Events (Vital Signs) (Optimized / Stream-to-Disk)
# ==============================================================================

print("\n" + "="*80)
print("STEP 7: Loading ICU Chart Events Data (Chunked & Column-Optimized)")
print("="*80)

# Define key vital sign itemids and support flags (Moved from Cell 13)
vital_sign_items = {
    220045: 'Heart Rate',
    220050: 'Arterial Blood Pressure systolic',
    220051: 'Arterial Blood Pressure diastolic',
    220052: 'Arterial Blood Pressure mean',
    220179: 'Non Invasive Blood Pressure systolic',
    220180: 'Non Invasive Blood Pressure diastolic',
    220181: 'Non Invasive Blood Pressure mean',
    220210: 'Respiratory Rate',
    223761: 'Temperature Fahrenheit',
    223762: 'Temperature Celsius',
    220277: 'SpO2'
}

oxygen_itemids = [223834, 223835]  # O2 flow / device
vent_itemids = [720, 224684, 224685]  # vent mode / tidal vol / PEEP

# Collect all itemids we care about
relevant_chart_items = set(vital_sign_items.keys())
relevant_chart_items.update(oxygen_itemids)
relevant_chart_items.update(vent_itemids)

print(f"Defining {len(relevant_chart_items)} relevant chart items for filtering (Vitals, O2, Vent).")

# Load bucketed chart events
print(f"Loading chartevents (chunked)...", end=" ")
chunk_size = 100000  # Reduced to 100k to prevent memory spikes
chart_chunks = []
total_rows_scanned = 0

# Cols to keep - CRITICAL for memory
use_cols = ['stay_id', 'hadm_id', 'charttime', 'itemid', 'valuenum', 'valueuom']
dtypes = {
    'itemid': 'int32',
    'stay_id': 'float32', # Use float to handle potential NaNs safely during load
    'hadm_id': 'float32',
    'valuenum': 'float32'
}

# Temp file for filtered data
temp_chart_file = OUTPUT_PATH / "temp_chartevents_filtered.csv"
print(f"Streaming filtered data to: {temp_chart_file}")

# Initialize file with header
first_chunk = True

with pd.read_csv(ICU_PATH / "chartevents.csv.gz",
                 chunksize=chunk_size,
                 usecols=use_cols,
                 dtype=dtypes,
                 iterator=True) as reader:
    for chunk in reader:
        total_rows_scanned += len(chunk)

        # Filter
        filtered_chunk = chunk[chunk['itemid'].isin(relevant_chart_items)]

        if not filtered_chunk.empty:
            # Optimize types before write
            # Convert IDs back to Int32 (Simulated with float for safety if NaNs exist, but for output clean up)
             # Write to disk IMMEDIATELY
            filtered_chunk.to_csv(temp_chart_file,
                                mode='w' if first_chunk else 'a',
                                header=first_chunk,
                                index=False)
            first_chunk = False

        del chunk

print(f"\n✓ Chart events filtered & streamed to disk (scanned ~{total_rows_scanned:,} rows)")

# Now read back the MUCH smaller filtered file
# If this still crashes, your dataset is too large for your RAM even after filtering.
print("Loading back filtered subset...")
if temp_chart_file.exists():
    chartevents = pd.read_csv(temp_chart_file, dtype=dtypes)
    # Convert IDs to integer where possible (ignoring NaNs)
    for col in ['stay_id', 'hadm_id']:
        chartevents[col] = pd.to_numeric(chartevents[col], errors='coerce').astype('Int32')
else:
    chartevents = pd.DataFrame(columns=use_cols)

print(f"\n✓ Chart events loaded & filtered: {len(chartevents):,} relevant records (from ~{total_rows_scanned:,} total)")

# Clean up
del chart_chunks
gc.collect()

print(f"Loading d_items...", end=" ")
d_items = pd.read_csv(ICU_PATH / "d_items.csv.gz")
print(f"\n✓ Item dictionary loaded: {len(d_items):,} unique items")

# Merge with item descriptions
chartevents_merged = chartevents.merge(
    d_items,
    on='itemid',
    how='left'
)

# Convert datetime
chartevents_merged['charttime'] = pd.to_datetime(chartevents_merged['charttime'])

print(f"\nChart Events Statistics (Filtered):")
print(f"- Total unique item types: {chartevents_merged['itemid'].nunique()}")
print(f"- Total unique ICU stays with chart events: {chartevents_merged['stay_id'].nunique()}")
print("\n✓ Cell 9 Complete: Chart events data loaded")


STEP 7: Loading ICU Chart Events Data (Chunked & Column-Optimized)
Defining 16 relevant chart items for filtering (Vitals, O2, Vent).
Loading chartevents (chunked)... Streaming filtered data to: output/temp_chartevents_filtered.csv

✓ Chart events filtered & streamed to disk (scanned ~432,997,491 rows)
Loading back filtered subset...

✓ Chart events loaded & filtered: 56,968,539 relevant records (from ~432,997,491 total)
Loading d_items... 
✓ Item dictionary loaded: 4,095 unique items

Chart Events Statistics (Filtered):
- Total unique item types: 15
- Total unique ICU stays with chart events: 93343

✓ Cell 9 Complete: Chart events data loaded


In [11]:
# ==============================================================================
# CELL 10: Load ICU Input/Output Events
# ==============================================================================

print("\n" + "="*80)
print("STEP 8: Loading ICU Input/Output Events")
print("="*80)

# Load input events
print(f"Loading inputevents...", end=" ")
inputevents = pd.read_csv(ICU_PATH / "inputevents.csv.gz")
print(f"\n✓ Input events loaded: {len(inputevents):,} records")

# Load output events
print(f"Loading outputevents...", end=" ")
outputevents = pd.read_csv(ICU_PATH / "outputevents.csv.gz")
print(f"✓ Output events loaded: {len(outputevents):,} records")

# Convert datetime columns
inputevents['starttime'] = pd.to_datetime(inputevents['starttime'])
inputevents['endtime'] = pd.to_datetime(inputevents['endtime'])
outputevents['charttime'] = pd.to_datetime(outputevents['charttime'])

print(f"\nInput Events Statistics:")
print(f"- Total unique ICU stays with inputs: {inputevents['stay_id'].nunique()}")
print(f"- Total unique input item types: {inputevents['itemid'].nunique()}")

print(f"\nOutput Events Statistics:")
print(f"- Total unique ICU stays with outputs: {outputevents['stay_id'].nunique()}")
print(f"- Total unique output item types: {outputevents['itemid'].nunique()}")
print("\n✓ Cell 10 Complete: Input/output events loaded")


STEP 8: Loading ICU Input/Output Events
Loading inputevents... 
✓ Input events loaded: 10,953,713 records
Loading outputevents... ✓ Output events loaded: 5,359,395 records

Input Events Statistics:
- Total unique ICU stays with inputs: 84345
- Total unique input item types: 327

Output Events Statistics:
- Total unique ICU stays with outputs: 91510
- Total unique output item types: 71

✓ Cell 10 Complete: Input/output events loaded


In [12]:
# ==============================================================================
# CELL 11: Create Base Admission Summary
# ==============================================================================

print("\n" + "="*80)
print("STEP 9: Creating Base Admission Summary")
print("="*80)

# Start with admissions as the base
admission_summary = admissions.copy()

# Merge with patient demographics
admission_summary = admission_summary.merge(
    patients[['subject_id', 'gender', 'anchor_age']],
    on='subject_id',
    how='left'
)

print(f"\n✓ Base admission summary created with {len(admission_summary)} admissions")
print("\n✓ Cell 11 Complete: Base admission summary created")


STEP 9: Creating Base Admission Summary

✓ Base admission summary created with 546028 admissions

✓ Cell 11 Complete: Base admission summary created


In [13]:
# ==============================================================================
# CELL 12: Extract ICD Comorbidity Flags (Optimized)
# ==============================================================================

print("\n" + "="*80)
print("STEP 10: Extracting Comorbidity Flags from ICD Codes (Pre-calculated)")
print("="*80)

# MEMORY OPTIMIZATION:
# Comorbidity flags were calculated on-the-fly in Cell 4 and stored in 'comorbidity_summary'.
# We just merge them here.

if 'comorbidity_summary' not in locals():
    print("WARNING: comorbidity_summary not found. Did Cell 4 run correctly?")
    comorbidity_summary = pd.DataFrame(columns=['hadm_id'])

print("Merging pre-calculated flags into admission summary...")
admission_summary = admission_summary.merge(
    comorbidity_summary,
    on="hadm_id",
    how="left"
)

# Ensure all comorbidity_cols exist in admission_summary before filling NaNs
# (comorbidity_cols defined in Cell 4)
for col in comorbidity_cols:
    if col not in admission_summary.columns:
        admission_summary[col] = False

admission_summary[comorbidity_cols] = admission_summary[comorbidity_cols].fillna(False)

print(f"\n✓ Comorbidity flags extracted and merged into admission_summary")
print(f"- Comorbidity columns added: {len(comorbidity_cols)}")
print("\n✓ Cell 12 Complete: Comorbidity flags merged")


STEP 10: Extracting Comorbidity Flags from ICD Codes (Pre-calculated)
Merging pre-calculated flags into admission summary...

✓ Comorbidity flags extracted and merged into admission_summary
- Comorbidity columns added: 10

✓ Cell 12 Complete: Comorbidity flags merged


In [14]:
# ==============================================================================
# CELL 13: Extract Vital Signs and Organ Support Flags (Optimized)
# ==============================================================================

# Note: this cell originally used 'chartevents_merged', 'procedures_merged', and 'prescriptions'
# We now use the optimized loading from Cell 7/9 and pre-calculated flags from Cell 5/6.

print("\n" + "="*80)
print("STEP 11: Extracting Key Vital Signs and Organ Support Flags (Optimized)")
print("="*80)

# Vital Signs Extraction (Already done in Cell 9 logic, but we need to summarize for admission_summary)
# In Cell 9 we loaded 'chartevents' (filtered).
# We need to ensure Cell 9 ran and populated 'chartevents' dataframe.

if 'chartevents' not in locals():
    print("WARNING: 'chartevents' dataframe missing. Cell 9 must run first.")
    # Fallback to empty
    chartevents = pd.DataFrame(columns=['stay_id', 'itemid', 'valuenum'])

# Recalculate vital_sign_items dict keys if needed, but they are defined at top of script or Cell 9
# We'll just re-map based on the dataframe we have.

# Filter chartevents for vital signs (using itemids present in data)
# Note: chartevents now ONLY contains relevant items, so we can just use it directly or filter if we want specific subset
# But wait, Cell 9 filtered for BOTH vitals and O2/Vent. We need to slit them.

# Re-define logic briefly for clarity (or reuse variables if scope allows)
vital_sign_items = {
    220045: 'Heart Rate', 220050: 'Arterial Blood Pressure systolic', 220051: 'Arterial Blood Pressure diastolic',
    220052: 'Arterial Blood Pressure mean', 220179: 'Non Invasive Blood Pressure systolic',
    220180: 'Non Invasive Blood Pressure diastolic', 220181: 'Non Invasive Blood Pressure mean',
    220210: 'Respiratory Rate', 223761: 'Temperature Fahrenheit', 223762: 'Temperature Celsius', 220277: 'SpO2'
}
oxygen_itemids = [223834, 223835]
vent_itemids = [720, 224684, 224685]

# Separate Vitals
vital_signs_df = chartevents[chartevents['itemid'].isin(vital_sign_items.keys())].copy()
vital_signs_df['vital_sign'] = vital_signs_df['itemid'].map(vital_sign_items)

# Create vital signs pivot
vital_pivot = vital_signs_df.groupby(['stay_id', 'vital_sign'])['valuenum'].agg(['mean', 'min', 'max']).reset_index()
print(f"✓ Calculated summary for {len(vital_signs_df):,} vital sign measurements")

# Extract Oxygen/Vent from chartevents
print("Extracting oxygen/vent flags from chartevents...")
support_ce = chartevents[chartevents['itemid'].isin(oxygen_itemids + vent_itemids)]

# We need badm_id for this. chartevents has it (we added it to usecols!).
if 'hadm_id' not in support_ce.columns:
    # If using verify strict mode and missed it, this would error.
    # But we fixed it in verification step 75.
    print("WARNING: hadm_id missing in chartevents. Cannot link O2/Vent to admission.")
    support_flags = pd.DataFrame(columns=['hadm_id', 'on_oxygen', 'on_ventilator'])
else:
    support_flags = support_ce.groupby("hadm_id")["itemid"].agg(list).reset_index()
    support_flags["on_oxygen"] = support_flags["itemid"].apply(lambda x: any(i in oxygen_itemids for i in x))
    support_flags["on_ventilator"] = support_flags["itemid"].apply(lambda x: any(i in vent_itemids for i in x))
    support_flags = support_flags[["hadm_id", "on_oxygen", "on_ventilator"]]

# Dialysis (from Cell 5 pre-calc)
if 'dialysis_flags' not in locals():
    dialysis_flags = pd.DataFrame(columns=['hadm_id', 'on_dialysis'])

# Vasopressors (from Cell 6 pre-calc)
if 'pressor_flags' not in locals():
    pressor_flags = pd.DataFrame(columns=['hadm_id', 'on_vasopressors'])

# Merge all support flags into admission_summary
print("Merging organ support flags...")
admission_summary = admission_summary.merge(support_flags, on="hadm_id", how="left")
admission_summary = admission_summary.merge(dialysis_flags, on="hadm_id", how="left")
admission_summary = admission_summary.merge(pressor_flags, on="hadm_id", how="left")

admission_summary[["on_oxygen", "on_ventilator", "on_dialysis", "on_vasopressors"]] = admission_summary[["on_oxygen", "on_ventilator", "on_dialysis", "on_vasopressors"]].fillna(False)

print(f"✓ Organ support flags extracted and merged")
print("\n✓ Cell 13 Complete: Vital signs and organ support flags extracted")


STEP 11: Extracting Key Vital Signs and Organ Support Flags (Optimized)
✓ Calculated summary for 53,806,854 vital sign measurements
Extracting oxygen/vent flags from chartevents...
Merging organ support flags...
✓ Organ support flags extracted and merged

✓ Cell 13 Complete: Vital signs and organ support flags extracted


In [15]:
# ==============================================================================
# CELL 14: Extract Key Lab Results and Liver Labs
# ==============================================================================

print("\n" + "="*80)
print("STEP 12: Extracting Key Lab Results")
print("="*80)

# Define key lab test itemids - ALREADY DEFINED IN CELL 7
# key_lab_items = { ... }

# Filter labevents for key labs
key_labs = labevents_merged[labevents_merged['itemid'].isin(key_lab_items.keys())].copy()
key_labs['lab_name'] = key_labs['itemid'].map(key_lab_items)

print(f"\n✓ Extracted {len(key_labs):,} key lab test results")

# Create first and last lab values per admission
first_labs = key_labs.sort_values('charttime').groupby(['hadm_id', 'lab_name']).first()['valuenum'].reset_index()
first_labs.columns = ['hadm_id', 'lab_name', 'first_value']

last_labs = key_labs.sort_values('charttime').groupby(['hadm_id', 'lab_name']).last()['valuenum'].reset_index()
last_labs.columns = ['hadm_id', 'lab_name', 'last_value']

lab_summary = first_labs.merge(last_labs, on=['hadm_id', 'lab_name'], how='outer')

# Extract Liver Lab Values
print("\nExtracting liver lab values...")

# liver_lab_map ALREADY DEFINED IN CELL 7
# liver_lab_map = { ... }

liver_lab_frames = []
for lab, itemids in liver_lab_map.items():
    tmp = labevents_merged[labevents_merged["itemid"].isin(itemids)]
    tmp = tmp.groupby("hadm_id")["valuenum"].agg(["first", "last"]).reset_index()
    tmp.columns = ["hadm_id", f"{lab}_first", f"{lab}_last"]
    liver_lab_frames.append(tmp)

liver_lab_summary = liver_lab_frames[0]
for df in liver_lab_frames[1:]:
    liver_lab_summary = liver_lab_summary.merge(df, on="hadm_id", how="outer")

# Merge into admission summary
admission_summary = admission_summary.merge(
    liver_lab_summary,
    on="hadm_id",
    how="left"
)

print(f"✓ Liver lab values extracted and merged")
print("\n✓ Cell 14 Complete: Lab results extracted")


STEP 12: Extracting Key Lab Results

✓ Extracted 62,072,696 key lab test results

Extracting liver lab values...
✓ Liver lab values extracted and merged

✓ Cell 14 Complete: Lab results extracted


In [16]:
# ==============================================================================
# CELL 15: Save All Preprocessed Data
# ==============================================================================

print("\n" + "="*80)
print("STEP 13: Saving Preprocessed Data")
print("="*80)

# Save patient-level summary (if created earlier - optional)
# Note: The original script had a patient_summary, but we focused on admission_summary
# If you need patient_summary, uncomment and create it

# Save admission-level summary
admission_summary.to_csv(OUTPUT_PATH / "mimic_admission_summary.csv", index=False)
print(f"\n✓ Saved: mimic_admission_summary.csv ({len(admission_summary)} records)")

# The following files were ALREADY saved via Stream-to-Disk in Cells 4, 5, 6:
print(f"✓ Already Saved (Streamed): mimic_diagnoses_detailed.csv")
print(f"✓ Already Saved (Streamed): mimic_procedures_detailed.csv")
print(f"✓ Already Saved (Streamed): mimic_prescriptions.csv")

# Save key lab results
key_labs.to_csv(OUTPUT_PATH / "mimic_key_labs.csv", index=False)
print(f"✓ Saved: mimic_key_labs.csv ({len(key_labs)} records)")

# Save lab summary
lab_summary.to_csv(OUTPUT_PATH / "mimic_lab_summary.csv", index=False)
print(f"✓ Saved: mimic_lab_summary.csv ({len(lab_summary)} records)")

# Save ICU stays
icustays.to_csv(OUTPUT_PATH / "mimic_icustays.csv", index=False)
print(f"✓ Saved: mimic_icustays.csv ({len(icustays)} records)")

# Save vital signs
# 'vital_signs' dataframe might not exist if we used 'chartevents' directly in Cell 13 logic
# But we did create 'vital_signs_df' in Cell 13.
if 'vital_signs_df' in locals():
    vital_signs_df.to_csv(OUTPUT_PATH / "mimic_vital_signs.csv", index=False)
    print(f"✓ Saved: mimic_vital_signs.csv ({len(vital_signs_df)} records)")
elif 'vital_signs' in locals():
     vital_signs.to_csv(OUTPUT_PATH / "mimic_vital_signs.csv", index=False)
     print(f"✓ Saved: mimic_vital_signs.csv ({len(vital_signs)} records)")

# Save vital signs summary
if 'vital_pivot' in locals():
    vital_pivot.to_csv(OUTPUT_PATH / "mimic_vital_signs_summary.csv", index=False)
    print(f"✓ Saved: mimic_vital_signs_summary.csv ({len(vital_pivot)} records)")

# Export to Parquet format (faster loading)
print("\n" + "="*80)
print("Exporting to Parquet Format")
print("="*80)

PARQUET_PATH = OUTPUT_PATH / "parquet"
PARQUET_PATH.mkdir(exist_ok=True)

datasets = {
    'admission_summary': admission_summary,
    'icustays': icustays,
}
if 'labevents' in locals(): datasets['key_labs'] = labevents  # Note: key_labs variable might not be defined, check labevents or key_labs logic
if 'lab_summary' in locals(): datasets['lab_summary'] = lab_summary
if 'vital_signs_df' in locals(): datasets['vital_signs'] = vital_signs_df
elif 'vital_signs' in locals(): datasets['vital_signs'] = vital_signs
if 'vital_pivot' in locals(): datasets['vital_signs_summary'] = vital_pivot

print("Note: Huge streamed files (diagnoses, procedures, prescriptions) are CSV only to save RAM.")

for name, df in tqdm(datasets.items(), desc="Exporting to parquet"):
    parquet_file = PARQUET_PATH / f"mimic_{name}.parquet"
    df.to_parquet(parquet_file, index=False, compression='snappy')
    print(f"✓ Saved: {parquet_file.name} ({len(df)} records)")

print(f"\n✓ All files exported to parquet format in: {PARQUET_PATH}")
print("\nParquet files are 2-5x faster to load than CSV!")
print("To load: df = pd.read_parquet('output/parquet/mimic_admission_summary.parquet')")

print("\n" + "="*80)
print("PREPROCESSING COMPLETE!")
print("="*80)
print(f"\nAll preprocessed files saved to: {OUTPUT_PATH}")
print("\n✓ Cell 15 Complete: All data saved")



STEP 13: Saving Preprocessed Data

✓ Saved: mimic_admission_summary.csv (546028 records)
✓ Already Saved (Streamed): mimic_diagnoses_detailed.csv
✓ Already Saved (Streamed): mimic_procedures_detailed.csv
✓ Already Saved (Streamed): mimic_prescriptions.csv
✓ Saved: mimic_key_labs.csv (62072696 records)
✓ Saved: mimic_lab_summary.csv (6519446 records)
✓ Saved: mimic_icustays.csv (94458 records)
✓ Saved: mimic_vital_signs.csv (53806854 records)
✓ Saved: mimic_vital_signs_summary.csv (765510 records)

Exporting to Parquet Format
Note: Huge streamed files (diagnoses, procedures, prescriptions) are CSV only to save RAM.


Exporting to parquet:  17%|█▋        | 1/6 [00:02<00:12,  2.44s/it]

✓ Saved: mimic_admission_summary.parquet (546028 records)
✓ Saved: mimic_icustays.parquet (94458 records)


Exporting to parquet:  50%|█████     | 3/6 [00:16<00:18,  6.02s/it]

✓ Saved: mimic_key_labs.parquet (68901139 records)


Exporting to parquet:  67%|██████▋   | 4/6 [00:17<00:08,  4.27s/it]

✓ Saved: mimic_lab_summary.parquet (6519446 records)


Exporting to parquet: 100%|██████████| 6/6 [00:35<00:00,  5.87s/it]

✓ Saved: mimic_vital_signs.parquet (53806854 records)
✓ Saved: mimic_vital_signs_summary.parquet (765510 records)

✓ All files exported to parquet format in: output/parquet

Parquet files are 2-5x faster to load than CSV!
To load: df = pd.read_parquet('output/parquet/mimic_admission_summary.parquet')

PREPROCESSING COMPLETE!

All preprocessed files saved to: output

✓ Cell 15 Complete: All data saved


In [18]:
# ==============================================================================
# VALIDATION: Check Final admission_summary
# ==============================================================================

print("\n" + "="*80)
print("VALIDATION: Final Admission Summary Check")
print("="*80)

print(f"\nTotal admissions: {len(admission_summary)}")
print(f"Total columns: {len(admission_summary.columns)}")
print(f"\nColumn categories:")
print(f"- Comorbidity flags: {sum(col in admission_summary.columns for col in comorbidity_cols)}")
print(f"- Organ support flags: {sum(col in admission_summary.columns for col in ['on_oxygen', 'on_ventilator', 'on_dialysis', 'on_vasopressors'])}")
print(f"- Liver labs: {sum('alt_' in col or 'ast_' in col or 'alp_' in col or 'bilirubin' in col for col in admission_summary.columns)}")

print("\n✓ VALIDATION COMPLETE - Script executed successfully!")


VALIDATION: Final Admission Summary Check

Total admissions: 546028
Total columns: 41

Column categories:
- Comorbidity flags: 10
- Organ support flags: 4
- Liver labs: 8

✓ VALIDATION COMPLETE - Script executed successfully!
